## PART A - setup (always run, in this order)

**A1 setup**

In [ ]:
import os, sys, json, time, glob, re, shutil, platform, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return str(e)

print('TensorFlow', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print(sh('nvidia-smi -L'))

TensorFlow 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU 0: Tesla T4 (UUID: GPU-51c8377b-322a-ad60-b4f0-bc64c02f15b6)


**A2 upload and extract the data**

In [ ]:
# Upload archive.zip (the Kaggle dataset zip) when asked, then it is extracted to /content/data
from google.colab import files
import zipfile
files.upload()
os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile('/content/archive.zip') as z:
    z.extractall('/content/data')
print(sh('find /content/data -type f | head -60; du -sh /content/data'))

Saving archive.zip to archive.zip
/content/data/Portmap-training.parquet
/content/data/NetBIOS-training.parquet
/content/data/SNMP-testing.parquet
/content/data/UDP-testing.parquet
/content/data/Syn-testing.parquet
/content/data/UDPLag-training.parquet
/content/data/Syn-training.parquet
/content/data/LDAP-training.parquet
/content/data/MSSQL-testing.parquet
/content/data/MSSQL-training.parquet
/content/data/TFTP-testing.parquet
/content/data/LDAP-testing.parquet
/content/data/NetBIOS-testing.parquet
/content/data/NTP-testing.parquet
/content/data/UDP-training.parquet
/content/data/UDPLag-testing.parquet
/content/data/DNS-testing.parquet
34M	/content/data


**A3 configuration**

In [ ]:
CFG = dict(
    DATA_DIR='/content/data',
    PER_CLASS_CAP=10000,        # max rows kept per class per file chunk and per file (bounds RAM and time)
    SEEDS=[0, 1, 2],            # raise to 5 if time allows
    DROP_PORTS=True,            # ports are campaign-specific identifiers; set False to keep them
    USE_PCA=True, PCA_COMPONENTS=30,
    EPOCHS=10, BATCH=32, VAL_FRAC=0.1,
    TIMING_REPEATS=500, TIMING_BATCHES=[1, 256], TIMING_WARMUP=50,
    RUN_RANDOM_POOLED_COMPARISON=True,
    MODELS=['hybrid', 'dnn_gru', 'blstm'],
    FILE_DAY_OVERRIDE={},       # e.g. {'/content/data/xyz.parquet': 'train'} if a file cannot be assigned automatically
    OUT_DIR='/content/results',
)
os.makedirs(CFG['OUT_DIR'], exist_ok=True)

**A4 helper functions**

In [ ]:
ID_COLS = {'flow id', 'source ip', 'destination ip', 'src ip', 'dst ip', 'timestamp', 'unnamed: 0',
           'simillarhttp', 'similarhttp', 'flowid', 'srcip', 'dstip'}
PORT_COLS = {'source port', 'destination port', 'src port', 'dst port'}
META = ['label_raw', 'label', '_row', '_src', '_day', '_h']

def day_of(path):
    if path in CFG['FILE_DAY_OVERRIDE']:
        return CFG['FILE_DAY_OVERRIDE'][path]
    p = path.lower()
    if re.search(r'train|01-12|0112|01_12', p):
        return 'train'
    if re.search(r'test|03-11|0311|03_11', p):
        return 'test'
    return 'unknown'

def norm_label(s):
    s = str(s).strip().lower()
    s = re.sub(r'^drdos[_\- ]?', '', s)
    return re.sub(r'[^a-z0-9]', '', s)

def iter_chunks(f):
    off = 0
    if f.endswith('.parquet'):
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(f)
        gen = (b.to_pandas() for b in pf.iter_batches(batch_size=500000))
    else:
        gen = pd.read_csv(f, chunksize=500000, low_memory=False)
    for t in gen:
        t.columns = [str(c).strip() for c in t.columns]
        t['_row'] = np.arange(off, off + len(t))
        off += len(t)
        yield t

def cap_per_label(t, cap, rng):
    sel = []
    for lab, g in t.groupby('label'):
        idx = g.index.to_numpy()
        if len(idx) > cap:
            idx = rng.choice(idx, cap, replace=False)
        sel.append(idx)
    return t.loc[np.concatenate(sel)] if sel else t.iloc[:0]

def load_all():
    files_ = sorted(glob.glob(os.path.join(CFG['DATA_DIR'], '**', '*.parquet'), recursive=True) +
                    glob.glob(os.path.join(CFG['DATA_DIR'], '**', '*.csv'), recursive=True))
    rng = np.random.default_rng(12345)
    parts = []
    for f in files_:
        kept = []
        for t in iter_chunks(f):
            lc = next((c for c in t.columns if c.lower() == 'label'), None)
            if lc is None:
                raise ValueError('No Label column in %s: %s' % (f, list(t.columns)[:10]))
            t = t.rename(columns={lc: 'label_raw'})
            t['label'] = t['label_raw'].map(norm_label)
            kept.append(cap_per_label(t, CFG['PER_CLASS_CAP'], rng))
        if not kept:
            continue
        d = pd.concat(kept, ignore_index=True)
        d = cap_per_label(d, CFG['PER_CLASS_CAP'], rng).reset_index(drop=True)
        d['_src'] = os.path.relpath(f, CFG['DATA_DIR'])
        d['_day'] = day_of(f)
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

**A5 load data and check the day table**

In [ ]:
raw = load_all()
print('Rows loaded (capped):', len(raw))
print(raw.groupby('_day')['label'].value_counts().unstack(0).fillna(0).astype(int))
print(raw.groupby(['_src', '_day']).size())
assert 'unknown' not in set(raw['_day']), 'Some files were not assigned to train/test. Set CFG["FILE_DAY_OVERRIDE"].'
# >>> STOP: check the day assignment. In the Kaggle copy the *-training files hold the 7-attack set and the *-testing files the 12-attack set.

Rows loaded (capped): 159316
_day      test  train
label                
benign   35182  29393
dns       3669      0
ldap      1440   1906
mssql     6212   8523
netbios    598    644
ntp      10000      0
portmap      0    685
snmp      2717      0
syn        533  15538
tftp     10000      0
udp      10000  13298
udplag    8872     55
webddos     51      0
_src                      _day 
DNS-testing.parquet       test      6703
LDAP-testing.parquet      test      2831
LDAP-training.parquet     train     6715
MSSQL-testing.parquet     test      8083
MSSQL-training.parquet    train    10974
NTP-testing.parquet       test     20000
NetBIOS-testing.parquet   test      2225
NetBIOS-training.parquet  train     1631
Portmap-training.parquet  train     5105
SNMP-testing.parquet      test      4018
Syn-testing.parquet       test       907
Syn-training.parquet      train    20000
TFTP-testing.parquet      test     20000
UDP-testing.parquet       test     12042
UDP-training.parquet      train    

**A6 clean, deduplicate, define train/test**

In [ ]:
def clean(df):
    log = {'rows_in': int(len(df))}
    drop = [c for c in df.columns if c.lower() in ID_COLS or (CFG['DROP_PORTS'] and c.lower() in PORT_COLS)]
    feat = [c for c in df.columns if c not in META and c not in drop and pd.api.types.is_numeric_dtype(df[c])]
    X = df[feat].replace([np.inf, -np.inf], np.nan)
    ok = X.notna().all(axis=1)
    log['rows_removed_nan_or_inf'] = int((~ok).sum())
    df = df.loc[ok].copy()
    df[feat] = X.loc[ok].astype('float32')
    tr = df[df['_day'] == 'train']
    const = [c for c in feat if tr[c].nunique() <= 1]
    feat = [c for c in feat if c not in const]
    log['constant_features_dropped_using_train_day'] = const
    df['_h'] = pd.util.hash_pandas_object(df[feat], index=False).values
    log['hash_groups_with_conflicting_labels'] = int((df.groupby('_h')['label'].nunique() > 1).sum())
    n0 = len(df)
    df = df.sort_values('_day', ascending=False, kind='stable').drop_duplicates('_h', keep='first')
    log['rows_removed_exact_duplicates_incl_cross_day'] = int(n0 - len(df))
    log['rows_out'] = int(len(df))
    log['n_features'] = len(feat)
    return df.reset_index(drop=True), feat, log

df, FEAT, CLEAN_LOG = clean(raw)
print(json.dumps(CLEAN_LOG, indent=1))
print(df.groupby('_day')['label'].value_counts().unstack(0).fillna(0).astype(int))
assert 'benign' in set(df['label']), 'No benign class found: check label names'
train_df = df[df['_day'] == 'train'].reset_index(drop=True)
test_df = df[df['_day'] == 'test'].reset_index(drop=True)
print('train classes:', sorted(train_df['label'].unique()))
print('test classes :', sorted(test_df['label'].unique()))
print('classes only in test (unseen families):', sorted(set(test_df['label']) - set(train_df['label'])))

{
 "rows_in": 159316,
 "rows_removed_nan_or_inf": 0,
 "constant_features_dropped_using_train_day": [
  "Bwd PSH Flags",
  "Fwd URG Flags",
  "Bwd URG Flags",
  "FIN Flag Count",
  "PSH Flag Count",
  "ECE Flag Count",
  "Fwd Avg Bytes/Bulk",
  "Fwd Avg Packets/Bulk",
  "Fwd Avg Bulk Rate",
  "Bwd Avg Bytes/Bulk",
  "Bwd Avg Packets/Bulk",
  "Bwd Avg Bulk Rate"
 ],
 "hash_groups_with_conflicting_labels": 2487,
 "rows_removed_exact_duplicates_incl_cross_day": 10230,
 "rows_out": 149086,
 "n_features": 65
}
_day      test  train
label                
benign   34002  28444
dns       2837      0
ldap       683   1881
mssql     3335   8364
netbios    156    474
ntp       9997      0
portmap      0    437
snmp      1992      0
syn        533  15153
tftp      9989      0
udp       9362  12915
udplag    8445     37
webddos     50      0
train classes: ['benign', 'ldap', 'mssql', 'netbios', 'portmap', 'syn', 'udp', 'udplag']
test classes : ['benign', 'dns', 'ldap', 'mssql', 'netbios', 'ntp', 'sn

**A7 preprocessing and metrics**

In [ ]:
class Pipe:
    def fit(self, X):
        self.sc = MinMaxScaler().fit(X)
        Z = self.sc.transform(X)
        self.pca = None
        if CFG['USE_PCA']:
            self.pca = PCA(n_components=min(CFG['PCA_COMPONENTS'], Z.shape[1]), random_state=0).fit(Z)
        return self
    def transform(self, X):
        Z = self.sc.transform(X)
        if self.pca is not None:
            Z = self.pca.transform(Z)
        return Z.astype('float32')

def ece_score(conf, correct, bins=15):
    edges = np.linspace(0, 1, bins + 1)
    e, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf <= hi) if lo == 0 else (conf > lo) & (conf <= hi)
        if m.any():
            e += m.sum() / n * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def evaluate(y_names, proba, classes):
    classes = list(classes)
    ben = classes.index('benign')
    pred = np.array(classes)[proba.argmax(1)]
    out = {}
    yb = (y_names != 'benign').astype(int)
    pb = (pred != 'benign').astype(int)
    p, r, f, _ = precision_recall_fscore_support(yb, pb, average='binary', zero_division=0)
    patt = 1 - proba[:, ben]
    out.update(bin_acc=float(accuracy_score(yb, pb)), bin_precision=float(p), bin_recall=float(r), bin_f1=float(f),
               bin_fpr=float(((pb == 1) & (yb == 0)).sum() / max((yb == 0).sum(), 1)),
               bin_brier=float(np.mean((patt - yb) ** 2)),
               bin_ece=ece_score(np.where(patt >= 0.5, patt, 1 - patt), (pb == yb).astype(float)),
               n_test=int(len(yb)), n_test_attack=int(yb.sum()))
    out['bin_auc'] = float(roc_auc_score(yb, patt)) if 0 < yb.sum() < len(yb) else float('nan')
    seen = np.isin(y_names, classes)
    ys, ps, Ps = y_names[seen], pred[seen], proba[seen]
    out['mc_n'] = int(seen.sum())
    out['mc_acc'] = float(accuracy_score(ys, ps))
    labs = [c for c in classes if c in set(ys)]
    p, r, f, s = precision_recall_fscore_support(ys, ps, labels=labs, average=None, zero_division=0)
    out['mc_macro_p'], out['mc_macro_r'], out['mc_macro_f1'] = float(np.mean(p)), float(np.mean(r)), float(np.mean(f))
    out['per_class'] = {l: dict(precision=float(a), recall=float(b), f1=float(c), support=int(d))
                        for l, a, b, c, d in zip(labs, p, r, f, s)}
    out['confusion'] = {'labels': classes, 'matrix': confusion_matrix(ys, ps, labels=classes).tolist()}
    yi = np.array([classes.index(v) for v in ys])
    out['mc_brier'] = float(np.mean(np.sum((Ps - np.eye(len(classes))[yi]) ** 2, axis=1)))
    out['mc_ece'] = ece_score(Ps.max(1), (Ps.argmax(1) == yi).astype(float))
    out['unseen_family_detected_as_attack'] = {u: float(np.mean(pred[y_names == u] != 'benign'))
                                                for u in sorted(set(y_names[~seen]))}
    return out

**A8 models**

In [ ]:
from tensorflow.keras import layers as L

def build_hybrid(nf, nc):
    # RECONSTRUCTION from recorded unit counts: DNN-LSTM(500), DNN-LSTM(300), DNN-GRU(200), ReLU, softmax.
    i = tf.keras.Input((1, nf))
    x = L.Dense(500, activation='relu')(i); x = L.LSTM(500, return_sequences=True)(x)
    x = L.Dense(300, activation='relu')(x); x = L.LSTM(300, return_sequences=True)(x)
    x = L.Dense(200, activation='relu')(x); x = L.GRU(200)(x)
    return tf.keras.Model(i, L.Dense(nc, activation='softmax')(x), name='hybrid')

def build_dnn_gru(nf, nc):
    # Baseline defined here (original not recorded): Dense 500-300-200 (ReLU) then GRU(200).
    i = tf.keras.Input((1, nf))
    x = L.Dense(500, activation='relu')(i); x = L.Dense(300, activation='relu')(x)
    x = L.Dense(200, activation='relu')(x); x = L.GRU(200)(x)
    return tf.keras.Model(i, L.Dense(nc, activation='softmax')(x), name='dnn_gru')

def build_blstm(nf, nc):
    # Baseline defined here (original not recorded): Bidirectional LSTM(200) then Dense(100, ReLU).
    i = tf.keras.Input((1, nf))
    x = L.Bidirectional(L.LSTM(200))(i); x = L.Dense(100, activation='relu')(x)
    return tf.keras.Model(i, L.Dense(nc, activation='softmax')(x), name='blstm')

BUILDERS = {'hybrid': build_hybrid, 'dnn_gru': build_dnn_gru, 'blstm': build_blstm}

def fit_model(model, Xtr, ytr, Xva, yva, seed):
    tf.keras.utils.set_random_seed(seed)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy')
    cb = [tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)]
    t0 = time.perf_counter()
    h = model.fit(Xtr[:, None, :], ytr, validation_data=(Xva[:, None, :], yva), epochs=CFG['EPOCHS'],
                  batch_size=CFG['BATCH'], callbacks=cb, verbose=0)
    return time.perf_counter() - t0, len(h.history['loss'])

def predict_proba(model, X):
    return model.predict(X[:, None, :], batch_size=4096, verbose=0)

def time_model(model, X, batch, repeats, warmup):
    xb = tf.constant(X[:batch][:, None, :].astype('float32'))
    f = tf.function(lambda x: model(x, training=False))
    for _ in range(warmup):
        f(xb).numpy()
    ts = []
    for _ in range(repeats):
        t = time.perf_counter(); f(xb).numpy(); ts.append((time.perf_counter() - t) * 1000)
    return dict(batch=batch, median_ms=float(np.median(ts)), p95_ms=float(np.percentile(ts, 95)), mean_ms=float(np.mean(ts)))

**A9 run_one**

In [ ]:
def run_one(pname, mname, seed, tr, te, feat, do_timing=False):
    classes = sorted(tr['label'].unique())
    cidx = {c: i for i, c in enumerate(classes)}
    ytr_all = tr['label'].map(cidx).values
    idx = np.arange(len(tr))
    strat = ytr_all if np.bincount(ytr_all).min() >= 2 else None
    i_tr, i_va = train_test_split(idx, test_size=CFG['VAL_FRAC'], stratify=strat, random_state=seed)
    pipe = Pipe().fit(tr.iloc[i_tr][feat].values)          # scaler/PCA fitted on training rows only
    Xtr = pipe.transform(tr.iloc[i_tr][feat].values); ytr = ytr_all[i_tr]
    Xva = pipe.transform(tr.iloc[i_va][feat].values); yva = ytr_all[i_va]
    Xte = pipe.transform(te[feat].values); yte_names = te['label'].values
    model = BUILDERS[mname](Xtr.shape[1], len(classes))
    train_s, n_ep = fit_model(model, Xtr, ytr, Xva, yva, seed)
    proba = predict_proba(model, Xte)
    res = evaluate(yte_names, proba, classes)
    res.update(protocol=pname, model=mname, seed=seed, train_seconds=train_s, epochs_run=n_ep,
               params=int(model.count_params()), n_train=int(len(i_tr)), n_features_in=int(Xtr.shape[1]))
    if do_timing:
        res['timing_ms'] = [time_model(model, Xte, b, CFG['TIMING_REPEATS'], CFG['TIMING_WARMUP'])
                            for b in CFG['TIMING_BATCHES']]
        raw_te = te[feat].values[:256]
        ts = []
        for _ in range(CFG['TIMING_REPEATS']):
            t = time.perf_counter(); pipe.transform(raw_te[:1]); ts.append((time.perf_counter() - t) * 1000)
        res['preprocess_ms_batch1_median'] = float(np.median(ts))
    return res, proba, classes, yte_names

## PART B - EITHER retrain (about 1 hour: B1-B3) OR restore the earlier results (B4). Cross-validation (Part D) needs neither.

**B1 day-wise and random-split training loop**

In [ ]:
PROTOCOLS = {'daywise': (train_df, test_df)}
if CFG['RUN_RANDOM_POOLED_COMPARISON']:
    vc = df['label'].value_counts()
    pooled = df[df['label'].isin(vc[vc >= 2].index)].reset_index(drop=True)
    ptr, pte = train_test_split(pooled, test_size=0.30, stratify=pooled['label'], random_state=0)
    PROTOCOLS['random_pooled_70_30'] = (ptr.reset_index(drop=True), pte.reset_index(drop=True))

RESULTS, PROBS = [], {}
for pname, (tr, te) in PROTOCOLS.items():
    for mname in CFG['MODELS']:
        for seed in CFG['SEEDS']:
            first = (seed == CFG['SEEDS'][0]) and pname == 'daywise'
            res, proba, classes, yn = run_one(pname, mname, seed, tr, te, FEAT, do_timing=first)
            RESULTS.append(res)
            if seed == CFG['SEEDS'][0]:
                PROBS['%s_%s' % (pname, mname)] = dict(proba=proba.astype('float32'), y=yn, classes=np.array(classes))
            print(pname, mname, 'seed', seed, 'bin_acc %.4f bin_f1 %.4f mc_acc %.4f train_s %.0f'
                  % (res['bin_acc'], res['bin_f1'], res['mc_acc'], res['train_seconds']))

daywise hybrid seed 0 bin_acc 0.9785 bin_f1 0.9812 mc_acc 0.8439 train_s 700
daywise hybrid seed 1 bin_acc 0.9940 bin_f1 0.9948 mc_acc 0.8444 train_s 729
daywise hybrid seed 2 bin_acc 0.9923 bin_f1 0.9934 mc_acc 0.8433 train_s 635
daywise dnn_gru seed 0 bin_acc 0.9942 bin_f1 0.9950 mc_acc 0.8438 train_s 246
daywise dnn_gru seed 1 bin_acc 0.9876 bin_f1 0.9893 mc_acc 0.8346 train_s 242
daywise dnn_gru seed 2 bin_acc 0.9932 bin_f1 0.9941 mc_acc 0.8409 train_s 194
daywise blstm seed 0 bin_acc 0.9912 bin_f1 0.9925 mc_acc 0.8392 train_s 125
daywise blstm seed 1 bin_acc 0.9926 bin_f1 0.9936 mc_acc 0.8408 train_s 123
daywise blstm seed 2 bin_acc 0.9899 bin_f1 0.9913 mc_acc 0.8351 train_s 121
random_pooled_70_30 hybrid seed 0 bin_acc 0.9955 bin_f1 0.9961 mc_acc 0.9289 train_s 863
random_pooled_70_30 hybrid seed 1 bin_acc 0.9956 bin_f1 0.9962 mc_acc 0.9222 train_s 700


**B2 summary**

In [ ]:
rows = []
for r in RESULTS:
    rows.append({k: v for k, v in r.items() if not isinstance(v, (dict, list))})
R = pd.DataFrame(rows)
keys = ['bin_acc', 'bin_precision', 'bin_recall', 'bin_f1', 'bin_fpr', 'bin_auc', 'bin_brier', 'bin_ece',
        'mc_acc', 'mc_macro_f1', 'mc_brier', 'mc_ece', 'train_seconds', 'params']
summary = R.groupby(['protocol', 'model'])[keys].agg(['mean', 'std'])
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 50)
print(summary.round(4).to_string())
print()
for r in RESULTS:
    if 'timing_ms' in r:
        print(r['protocol'], r['model'], r['timing_ms'], 'preprocess_ms(batch1)=%.3f' % r['preprocess_ms_batch1_median'])
print()
for r in RESULTS:
    if r['seed'] == CFG['SEEDS'][0] and r['unseen_family_detected_as_attack']:
        print(r['protocol'], r['model'], 'unseen-family detected as attack:', r['unseen_family_detected_as_attack'])

                            bin_acc         bin_precision         bin_recall          bin_f1         bin_fpr         bin_auc         bin_brier         bin_ece          mc_acc         mc_macro_f1         mc_brier          mc_ece         train_seconds              params     
                               mean     std          mean     std       mean     std    mean     std    mean     std    mean     std      mean     std    mean     std    mean     std        mean     std     mean     std    mean     std          mean      std       mean  std
protocol            model                                                                                                                                                                                                                                                         
daywise             blstm    0.9913  0.0014        0.9931  0.0025     0.9920  0.0003  0.9925  0.0012  0.0096  0.0035  0.9991  0.0001    0.0073  0.0005  0.0062  0.0014  0.8385 

**B3 save results and download zip**

In [ ]:
env = dict(tensorflow=tf.__version__, python=sys.version, platform=platform.platform(),
           gpu=sh('nvidia-smi --query-gpu=name,driver_version --format=csv,noheader'),
           cpu=sh('lscpu | grep "Model name"'), cfg=CFG, clean_log=CLEAN_LOG, features=FEAT,
           class_counts_train=train_df['label'].value_counts().to_dict(),
           class_counts_test=test_df['label'].value_counts().to_dict())
json.dump(env, open(CFG['OUT_DIR'] + '/environment.json', 'w'), indent=1, default=str)
json.dump(RESULTS, open(CFG['OUT_DIR'] + '/all_results.json', 'w'), indent=1, default=str)
summary.to_csv(CFG['OUT_DIR'] + '/summary.csv')
for k, v in PROBS.items():
    np.savez_compressed('%s/probs_%s.npz' % (CFG['OUT_DIR'], k), **v)
df[['_src', '_row', '_day', 'label']].to_csv(CFG['OUT_DIR'] + '/row_manifest.csv', index=False)
shutil.make_archive('/content/ddos_results', 'zip', CFG['OUT_DIR'])
from google.colab import files
files.download('/content/ddos_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**B4 (instead of B1-B3) restore earlier results**

In [ ]:
# Restore the earlier results instead of retraining: upload ddos_results.zip when asked
from google.colab import files
import zipfile
files.upload()
with zipfile.ZipFile('/content/ddos_results.zip') as z:
    z.extractall(CFG['OUT_DIR'])
print(sorted(os.listdir(CFG['OUT_DIR'])))

## PART C - optional extras (need the results files from Part B)

**C1 prerequisite check**

In [ ]:
need = ['train_df', 'test_df', 'FEAT', 'CFG', 'Pipe', 'BUILDERS', 'fit_model']
missing = [n for n in need if n not in globals()]
print('Missing variables:', missing if missing else 'none')
print('Saved files:', sorted(os.listdir(CFG['OUT_DIR'])) if os.path.isdir(CFG['OUT_DIR']) else 'results dir not found')

**C2 calibration, per-class tables, confusion matrices**

In [ ]:
# Calibration, per-class and confusion-matrix analysis from the saved results.
import json, os, sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt
RES = '/content/results'
OUT = '/content/results/analysis'
os.makedirs(OUT, exist_ok=True)
R = json.load(open(os.path.join(RES, 'all_results.json')))
MODELS = ['hybrid', 'dnn_gru', 'blstm']; PROTOS = ['daywise', 'random_pooled_70_30']
def load(p, m):
    z = np.load(os.path.join(RES, 'probs_%s_%s.npz' % (p, m)), allow_pickle=True)
    return z['proba'], z['y'], list(z['classes'])
def rel_table(conf, correct, bins=10):
    edges = np.linspace(0, 1, bins + 1); rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mk = (conf >= lo) & (conf <= hi) if lo == 0 else (conf > lo) & (conf <= hi)
        if mk.any(): rows.append((conf[mk].mean(), correct[mk].mean(), int(mk.sum())))
    return rows
tabs, summ = [], []
fig, ax = plt.subplots(2, 2, figsize=(10, 9))
for pi, p in enumerate(PROTOS):
    for m in MODELS:
        pr, y, cl = load(p, m); b = cl.index('benign'); y = np.asarray(y).astype(str)
        yb = (y != 'benign').astype(int); patt = 1 - pr[:, b]
        pred_b = (patt >= .5).astype(int)
        confb = np.where(patt >= .5, patt, 1 - patt); corrb = (pred_b == yb).astype(float)
        seen = np.isin(y, cl); yi = np.array([cl.index(v) for v in y[seen]]); Ps = pr[seen]
        confm = Ps.max(1); corrm = (Ps.argmax(1) == yi).astype(float)
        for kind, conf, corr in (('binary', confb, corrb), ('multiclass_top_label', confm, corrm)):
            for mc, ac, n in rel_table(conf, corr): tabs.append((p, m, kind, mc, ac, n))
            summ.append((p, m, kind, conf.mean(), corr.mean(), conf.mean() - corr.mean()))
        for k, (conf, corr) in enumerate(((confb, corrb), (confm, corrm))):
            t = rel_table(conf, corr); ax[pi, k].plot([c for c, a, n in t], [a for c, a, n in t], marker='o', label=m)
for pi in range(2):
    for k in range(2):
        ax[pi, k].plot([0, 1], [0, 1], 'k--', lw=1); ax[pi, k].set_xlabel('mean confidence'); ax[pi, k].set_ylabel('observed accuracy')
        ax[pi, k].set_title('%s - %s' % (PROTOS[pi], ['binary', 'multiclass top-label'][k])); ax[pi, k].legend()
        ax[pi, k].set_xlim(0.4 if k else 0.5, 1.0); ax[pi, k].set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'calibration_reliability.png'), dpi=140); plt.close()
pd.DataFrame(tabs, columns=['protocol', 'model', 'kind', 'mean_conf', 'accuracy', 'n']).to_csv(os.path.join(OUT, 'reliability_table.csv'), index=False)
S = pd.DataFrame(summ, columns=['protocol', 'model', 'kind', 'mean_confidence', 'accuracy', 'overconfidence_gap'])
S.to_csv(os.path.join(OUT, 'confidence_vs_accuracy.csv'), index=False); print(S.round(4).to_string(index=False))
# per-class (mean over seeds)
rows = []
for r in R:
    for c, v in r['per_class'].items(): rows.append(dict(protocol=r['protocol'], model=r['model'], cls=c, **v))
P = pd.DataFrame(rows).groupby(['protocol', 'model', 'cls']).agg(precision=('precision', 'mean'), recall=('recall', 'mean'), f1=('f1', 'mean'),
                                                                support=('support', 'first'), recall_sd=('recall', 'std')).reset_index()
P.round(4).to_csv(os.path.join(OUT, 'per_class_results.csv'), index=False)
for p in PROTOS:
    print('\n', p, '- mean recall by class (hybrid / dnn_gru / blstm), support')
    w = P[P.protocol == p].pivot(index='cls', columns='model', values='recall')[MODELS]; w['support'] = P[(P.protocol == p) & (P.model == 'hybrid')].set_index('cls')['support']
    print(w.round(3).to_string())
# confusion matrices, seed 0, row-normalised
fig, ax = plt.subplots(1, 2, figsize=(15, 6.5))
for k, p in enumerate(PROTOS):
    r = [x for x in R if x['protocol'] == p and x['model'] == 'hybrid' and x['seed'] == 0][0]
    M = np.array(r['confusion']['matrix'], float); lab = r['confusion']['labels']
    keep = [i for i in range(len(lab)) if M[i].sum() > 0]; Mn = M[keep] / M[keep].sum(1, keepdims=True)
    ax[k].imshow(Mn, cmap='Blues', vmin=0, vmax=1); ax[k].set_xticks(range(len(lab))); ax[k].set_xticklabels(lab, rotation=60)
    ax[k].set_yticks(range(len(keep))); ax[k].set_yticklabels([lab[i] for i in keep]); ax[k].set_title('hybrid, %s (row-normalised)' % p)
    for i in range(len(keep)):
        for j in range(len(lab)):
            if Mn[i, j] >= .005: ax[k].text(j, i, '%.2f' % Mn[i, j], ha='center', va='center', fontsize=7)
    pd.DataFrame(M.astype(int), index=lab, columns=lab).to_csv(os.path.join(OUT, 'confusion_hybrid_%s.csv' % p))
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'confusion_matrices_hybrid.png'), dpi=140); plt.close()
print('\nwrote files to', OUT)

**C3 timing conventions (retrains one hybrid)**

In [ ]:
# ---- Timing under several measurement conventions (retrains ONE hybrid, day-wise, seed 0) ----
from sklearn.model_selection import train_test_split
classes = sorted(train_df['label'].unique()); cidx = {c: i for i, c in enumerate(classes)}
y_all = train_df['label'].map(cidx).values
i_tr, i_va = train_test_split(np.arange(len(train_df)), test_size=CFG['VAL_FRAC'],
                              stratify=y_all if np.bincount(y_all).min() >= 2 else None, random_state=0)
pipe = Pipe().fit(train_df.iloc[i_tr][FEAT].values)
Xtr = pipe.transform(train_df.iloc[i_tr][FEAT].values); Xva = pipe.transform(train_df.iloc[i_va][FEAT].values)
raw_te = test_df[FEAT].values; Xte = pipe.transform(raw_te)
model = BUILDERS['hybrid'](Xtr.shape[1], len(classes))
fit_model(model, Xtr, y_all[i_tr], Xva, y_all[i_va], 0)
pb = model.predict(Xte[:, None, :], batch_size=4096, verbose=0)
pred_attack = np.array(classes)[pb.argmax(1)] != 'benign'; true_attack = test_df['label'].values != 'benign'
print('sanity: binary accuracy of the retrained hybrid = %.4f (earlier seed-0 run: 0.9943)' % (pred_attack == true_attack).mean())

def bench(fn, repeats, warmup=20):
    for _ in range(warmup): fn()
    ts = []
    for _ in range(repeats):
        t = time.perf_counter(); fn(); ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts)), float(np.percentile(ts, 95)), float(np.mean(ts))

x1 = Xte[:1][:, None, :].astype('float32'); raw1 = raw_te[:1]; xfull = Xte[:, None, :].astype('float32')
rows = []
def add(device, label, fn, repeats, per_sample_n=1):
    med, p95, mean = bench(fn, repeats)
    rows.append(dict(device=device, convention=label, median_ms=med, p95_ms=p95, mean_ms=mean, per_sample_us=med * 1000 / per_sample_n))

f = tf.function(lambda x: model(x, training=False))
xc = tf.constant(x1)
add('GPU', 'tf.function, input already on device, batch 1', lambda: f(xc).numpy(), CFG['TIMING_REPEATS'])
add('GPU', 'tf.function, includes host->device input copy, batch 1', lambda: f(tf.constant(x1)).numpy(), CFG['TIMING_REPEATS'])
add('GPU', 'model.predict(), batch 1 (Keras call overhead)', lambda: model.predict(x1, verbose=0), 100)
add('GPU', 'model.predict_on_batch(), batch 1', lambda: model.predict_on_batch(x1), CFG['TIMING_REPEATS'])
add('GPU', 'preprocessing (scaler+PCA) + tf.function, one raw record', lambda: f(tf.constant(pipe.transform(raw1)[:, None, :])).numpy(), CFG['TIMING_REPEATS'])
add('GPU', 'model.predict() on the WHOLE test set (%d rows), total' % len(Xte), lambda: model.predict(xfull, batch_size=4096, verbose=0), 5, per_sample_n=len(Xte))
with tf.device('/CPU:0'):
    m_cpu = tf.keras.models.clone_model(model); m_cpu.set_weights(model.get_weights())
    fc = tf.function(lambda x: m_cpu(x, training=False)); xcc = tf.constant(x1)
    add('CPU', 'tf.function, batch 1', lambda: fc(xcc).numpy(), 200)
    add('CPU', 'model.predict(), batch 1', lambda: m_cpu.predict(x1, verbose=0), 50)
    add('CPU', 'model.predict() on the WHOLE test set, total', lambda: m_cpu.predict(xfull, batch_size=4096, verbose=0), 2, per_sample_n=len(Xte))
T = pd.DataFrame(rows)
print(T.round(3).to_string(index=False))
T.to_csv(CFG['OUT_DIR'] + '/timing_styles.csv', index=False)
print('\nGPU:', sh('nvidia-smi --query-gpu=name --format=csv,noheader'), '| CPU:', sh('lscpu | grep "Model name"'))
print('NOTE: none of these include feature extraction, controller communication or flow-rule installation.')

**C4 reproducibility package**

In [ ]:
# ---- Reproducibility package ----
import hashlib, zipfile
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for b in iter(lambda: fh.read(chunk), b''): h.update(b)
    return h.hexdigest()
PKG = '/content/repro_package'; shutil.rmtree(PKG, ignore_errors=True); os.makedirs(PKG)
for fn in os.listdir(CFG['OUT_DIR']):
    p = os.path.join(CFG['OUT_DIR'], fn)
    if os.path.isfile(p): shutil.copy(p, PKG)
if os.path.isdir(os.path.join(CFG['OUT_DIR'], 'analysis')):
    shutil.copytree(os.path.join(CFG['OUT_DIR'], 'analysis'), os.path.join(PKG, 'analysis'))
data_zip = '/content/archive.zip'
checks = {'archive.zip': sha256(data_zip)} if os.path.exists(data_zip) else {}
for fn in sorted(os.listdir(PKG)):
    p = os.path.join(PKG, fn)
    if os.path.isfile(p): checks[fn] = sha256(p)
json.dump(checks, open(PKG + '/checksums_sha256.json', 'w'), indent=1)
versions = sh('pip list 2>/dev/null | grep -i -E "^(tensorflow|keras|numpy|pandas|scikit-learn|pyarrow|matplotlib) "')
open(PKG + '/requirements_snapshot.txt', 'w').write(versions + '\n')
readme = (
    'Reproducibility package: leakage-aware re-evaluation on CICDDoS2019 (Kaggle copy dhoogla/cicddos2019)\n\n'
    'Contents: all_results.json (every run, per-class metrics, confusion matrices, timing), summary.csv, environment.json\n'
    '(hardware, versions, configuration, cleaning log, class counts), row_manifest.csv (source file, original row, day, label for every row used),\n'
    'probs_*.npz (test-set probabilities, seed 0), analysis/ (calibration, per-class, confusion matrices), timing_styles.csv,\n'
    'checksums_sha256.json, requirements_snapshot.txt.\n\n'
    'Seeds: %s. Models are reconstructions from recorded unit counts (see the notebook).\n'
    'Split: files named *-training / *-testing in the Kaggle copy; exact duplicates removed (including cross-day); scaler/PCA fitted on training rows only.\n'
    'To reproduce: download the Kaggle dataset, run the notebook top to bottom on a T4 GPU runtime; add the notebook (.ipynb) to this package before depositing.\n'
    'Not included: controller/flow-rule measurements (not part of this experiment).\n' % CFG['SEEDS'])
open(PKG + '/README.txt', 'w').write(readme)
shutil.make_archive('/content/ddos_repro_package', 'zip', PKG)
print('Package written: /content/ddos_repro_package.zip; files:', sorted(os.listdir(PKG)))
try:
    from google.colab import files
    files.download('/content/ddos_repro_package.zip')
except ImportError:
    pass

## PART D - ten-fold cross-validation (needs only Part A)

**D1 configuration and check**

In [ ]:
CV = dict(FOLDS=10, MODELS=['hybrid', 'dnn_gru', 'blstm'], DESIGNS=['stratified', 'blocked'], GAP_FRAC=0.02,
          FILE=CFG['OUT_DIR'] + '/cv_results.json')
need = ['df', 'FEAT', 'Pipe', 'BUILDERS', 'fit_model', 'evaluate', 'run_one']
print('Missing variables:', [n for n in need if n not in globals()] or 'none')
from sklearn.model_selection import StratifiedKFold
import scipy.stats as sst

**D2 fold assignment and diagnostic**

In [ ]:
def stratified_folds(d, k, seed=0):
    fold = np.full(len(d), -1)
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    for f, (_, te) in enumerate(skf.split(np.zeros(len(d)), d['label'].values)):
        fold[te] = f
    return fold, None, None

def blocked_folds(d, k):
    fold = np.full(len(d), -1); rank = np.zeros(len(d), int); flen = np.zeros(len(d), int)
    rows = d['_row'].values
    for src, idx in d.groupby('_src').indices.items():
        order = idx[np.argsort(rows[idx], kind='stable')]
        n = len(order); r = np.arange(n)
        fold[order] = np.minimum((r * k) // n, k - 1); rank[order] = r; flen[order] = n
    return fold, rank, flen

def fold_split(d, fold, rank, flen, f, k, gap_frac):
    te = np.where(fold == f)[0]
    trm = fold != f
    if rank is not None:
        start = (f * flen + k - 1) // k; end = ((f + 1) * flen + k - 1) // k; g = np.ceil(gap_frac * flen).astype(int)
        purge = (rank >= start - g) & (rank < end + g)
        trm = trm & ~purge
    return np.where(trm)[0], te

# ---- diagnostic: is row order temporally structured? (lag-1 autocorrelation within files) ----
ac = []
for src, g in df.sort_values('_row').groupby('_src'):
    if len(g) < 500: continue
    for c in FEAT[:8]:
        x = g[c].values.astype('float64')
        if x.std() > 0: ac.append(np.corrcoef(x[:-1], x[1:])[0, 1])
print('mean lag-1 autocorrelation of features in original row order: %.3f (n=%d)' % (np.nanmean(ac), len(ac)))
print('  near 0 -> row order looks shuffled (blocking will behave like random folds); clearly > 0.1 -> temporal structure present')
FOLDMAP = {'stratified': stratified_folds(df, CV['FOLDS']), 'blocked': blocked_folds(df, CV['FOLDS'])}
for name, (fold, rank, flen) in FOLDMAP.items():
    print(name, 'test sizes per fold:', [int((fold == f).sum()) for f in range(CV['FOLDS'])])

**D3 run the folds (resumable)**

In [ ]:
import json as _json
RESULTS_CV = _json.load(open(CV['FILE'])) if os.path.exists(CV['FILE']) else []
done = {(r['protocol'], r['model'], r['fold']) for r in RESULTS_CV}
t_start = time.time(); n_new = 0
total = len(CV['DESIGNS']) * len(CV['MODELS']) * CV['FOLDS']
for design in CV['DESIGNS']:
    fold, rank, flen = FOLDMAP[design]
    for f in range(CV['FOLDS']):
        i_tr, i_te = fold_split(df, fold, rank, flen, f, CV['FOLDS'], CV['GAP_FRAC'])
        tr, te = df.iloc[i_tr].reset_index(drop=True), df.iloc[i_te].reset_index(drop=True)
        for m in CV['MODELS']:
            if (design, m, f) in done: continue
            res, _, _, _ = run_one(design, m, f, tr, te, FEAT, do_timing=False)
            res.update(fold=f, n_train_rows=int(len(tr)), n_test_rows=int(len(te)))
            RESULTS_CV.append(res); n_new += 1
            _json.dump(RESULTS_CV, open(CV['FILE'], 'w'), default=str)
            el = time.time() - t_start; left = total - len(RESULTS_CV)
            print('%s fold %d %s: bin_acc %.4f bin_f1 %.4f mc_acc %.4f | done %d/%d, est. %.0f min left'
                  % (design, f, m, res['bin_acc'], res['bin_f1'], res['mc_acc'], len(RESULTS_CV), total, el / n_new * left / 60))

**D4 aggregate, paired tests, plots**

In [ ]:
import matplotlib.pyplot as plt
rows = [{k: v for k, v in r.items() if not isinstance(v, (dict, list))} for r in RESULTS_CV]
C = pd.DataFrame(rows)
metrics = ['bin_acc', 'bin_precision', 'bin_recall', 'bin_f1', 'bin_fpr', 'bin_auc', 'mc_acc', 'mc_macro_f1', 'mc_ece', 'bin_ece']
def tci(x):
    x = np.asarray(x, float); n = len(x); h = sst.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
    return x.mean() - h, x.mean() + h
out = []
for (d, m), g in C.groupby(['protocol', 'model']):
    for k in metrics:
        lo, hi = tci(g[k]); out.append(dict(design=d, model=m, metric=k, mean=g[k].mean(), sd=g[k].std(ddof=1),
                                           ci_low=lo, ci_high=hi, min=g[k].min(), max=g[k].max(), n_folds=len(g)))
S = pd.DataFrame(out); S.to_csv(CFG['OUT_DIR'] + '/cv_summary.csv', index=False)
print(S[S.metric.isin(['bin_acc', 'bin_f1', 'bin_fpr', 'mc_acc', 'mc_macro_f1'])].round(4).to_string(index=False))
C.to_csv(CFG['OUT_DIR'] + '/cv_fold_table.csv', index=False)
# paired fold comparisons: hybrid vs each baseline
print()
for d in CV['DESIGNS']:
    for other in [m for m in CV['MODELS'] if m != 'hybrid']:
        a = C[(C.protocol == d) & (C.model == 'hybrid')].sort_values('fold'); b = C[(C.protocol == d) & (C.model == other)].sort_values('fold')
        if len(a) == len(b) and len(a) > 2:
            for k in ['bin_acc', 'bin_f1', 'mc_acc']:
                diff = a[k].values - b[k].values
                print('%s hybrid-%s %s: mean diff %+.4f, paired t p=%.3f, Wilcoxon p=%.3f, hybrid better in %d/%d folds'
                      % (d, other, k, diff.mean(), sst.ttest_rel(a[k], b[k]).pvalue, sst.wilcoxon(diff).pvalue if np.any(diff) else 1.0, int((diff > 0).sum()), len(diff)))
# stratified vs blocked gap
print()
for m in CV['MODELS']:
    s = C[(C.protocol == 'stratified') & (C.model == m)]; b = C[(C.protocol == 'blocked') & (C.model == m)]
    if len(s) and len(b):
        print('%s: stratified minus blocked  bin_acc %+.4f, mc_acc %+.4f' % (m, s.bin_acc.mean() - b.bin_acc.mean(), s.mc_acc.mean() - b.mc_acc.mean()))
# per-class recall across folds
pc = []
for r in RESULTS_CV:
    for c, v in r['per_class'].items(): pc.append(dict(design=r['protocol'], model=r['model'], fold=r['fold'], cls=c, recall=v['recall'], precision=v['precision'], support=v['support']))
PC = pd.DataFrame(pc).groupby(['design', 'model', 'cls']).agg(recall_mean=('recall', 'mean'), recall_sd=('recall', 'std'),
        precision_mean=('precision', 'mean'), support_total=('support', 'sum')).reset_index()
PC.round(4).to_csv(CFG['OUT_DIR'] + '/cv_per_class.csv', index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k, met in enumerate(['bin_acc', 'bin_f1']):
    labels, data = [], []
    for d in CV['DESIGNS']:
        for m in CV['MODELS']:
            labels.append('%s\n%s' % (d[:5], m)); data.append(C[(C.protocol == d) & (C.model == m)][met].values)
    ax[k].boxplot(data); ax[k].set_xticks(range(1, len(labels) + 1)); ax[k].set_xticklabels(labels, fontsize=7); ax[k].set_title('fold-level ' + met)
plt.tight_layout(); plt.savefig(CFG['OUT_DIR'] + '/cv_boxplots.png', dpi=140); plt.show()